# Phase 2 — Random Forest training and policy evaluation

Trains a Random Forest to predict the offloading **target** (LOCAL / EDGE /
CLOUD) from the device-context features logged by `MetricsRecorder`, then
evaluates the rule-based policy it learned from against the two baselines.

**Input**: `../data/training.csv` (collected via the Phase 1 runbook)

**Outputs** (written to `../outputs/`):

| File | Contents |
|------|----------|
| `feature-importance.png` | Gini importance, baseline feature set |
| `confusion-matrix.png` | RF vs rule-based ground truth |
| `latency-by-policy.png` | Mean + p95 latency per policy per task |
| `estimator-accuracy.png` | Predicted vs actual latency |
| `energy-validation.png` | Energy model vs measured proxy |
| `regret-by-policy.png` | Approximate regret against a matched-condition oracle |
| `comparison-summary.md` | Numbers for the thesis chapter |
| `rf-model.json` | Deployable to `mobile/app/src/main/assets/` |

Run top-to-bottom. Sections 1–6 are data loading and integrity checks; 7–10 are
the classifier; 11–16 are the policy evaluation the thesis chapter draws on.

## 1 — Setup

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

DATA_PATH = Path('../data/training.csv')
OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

print('Setup ok')

## 2 — Load CSV

The schema is defined by `MetricsCsvFormat.HEADER` in the Android app. The
`REQUIRED_COLUMNS` check below fails loudly on a CSV written by an older build,
rather than silently producing `NaN` columns halfway through the analysis.

In [ ]:
REQUIRED_COLUMNS = [
    'timestamp_iso', 'task_id', 'task_name', 'target', 'fell_back', 'actual_ms',
    'result_bytes', 'error', 'rule', 'battery_percent', 'is_charging',
    'network_type', 'network_score', 'rtt_ms', 'bandwidth_mbps', 'cpu_percent',
    'is_stable', 'est_local_ms', 'est_remote_ms',
    'est_local_energy_mj', 'est_remote_energy_mj',
    'speedup', 'executed_at', 'server_exec_ms',
    'measured_power_mw', 'measured_energy_mj', 'input_size_bytes',
    'debug_overrides', 'reasoning',
]

df = pd.read_csv(DATA_PATH)

missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing:
    raise ValueError(
        f'training.csv is missing {missing}.\n'
        'This CSV was written by an older build. Re-run evaluation/collect_data.ps1 '
        'with the current app installed — MetricsRecorder archives the old file '
        'automatically when the schema changes.'
    )

# Normalise the boolean-ish columns, which arrive as the strings "true"/"false".
for col in ('fell_back', 'is_charging', 'is_stable'):
    df[col] = df[col].astype(str).str.lower().map({'true': True, 'false': False})

# LatencyEstimator uses Float.MAX_VALUE/4 (~8.5e37) as its remote-latency
# sentinel when the device is offline, so OFFLINE rows carry a placeholder in
# est_remote_ms and speedup rather than a real estimate. Trees split on order
# alone, so this is harmless for the classifier and the columns are left
# untouched — but it would dominate every mean, axis, and correlation below.
SENTINEL_MS = 1e9
df['remote_est_is_sentinel'] = df['est_remote_ms'] >= SENTINEL_MS
n_sentinel = int(df['remote_est_is_sentinel'].sum())

# collect_data.ps1 Session B forces remote_energy_mj=50 so LOW_BATTERY_OFFLOAD
# fires reliably. Those rows carry a synthetic estimator value, not a prediction,
# and must be excluded from any cost-model validation.
df['debug_overrides'] = df['debug_overrides'].fillna('').astype(str)
df['has_debug_override'] = df['debug_overrides'].str.len() > 0
n_override = int(df['has_debug_override'].sum())

print(f'Rows: {len(df)}')
print(f'Collected: {df.timestamp_iso.min()}  ->  {df.timestamp_iso.max()}')
if n_sentinel:
    print(f'Offline-sentinel rows (est_remote_ms >= 1e9): {n_sentinel} '
          f'({100*n_sentinel/len(df):.1f}%) — excluded from descriptive stats.')
if n_override:
    print(f'Debug-override rows: {n_override} ({100*n_override/len(df):.1f}%) '
          f'— excluded from estimator and energy validation.')
    print(df.loc[df['has_debug_override'], 'debug_overrides'].value_counts().to_string())
df.head()

## 3 — Exploratory data analysis

Sanity-check before training: is every rule represented, and does each numeric
feature actually vary? A feature with zero variance contributes nothing and
usually means a collector was broken during the session.

In [ ]:
print('--- Target distribution ---')
print(df['target'].value_counts(), '\n')

adaptive = df[~df['rule'].str.startswith(('FORCED_', 'ML_PREDICTED'))]
print('--- Rule distribution (adaptive rows only) ---')
print(adaptive['rule'].value_counts(), '\n')

print('--- Task distribution ---')
print(df['task_name'].value_counts(), '\n')

NUMERIC = ['battery_percent', 'network_score', 'rtt_ms', 'bandwidth_mbps',
           'cpu_percent', 'est_local_ms', 'est_remote_ms',
           'est_local_energy_mj', 'est_remote_energy_mj', 'speedup', 'actual_ms']

summary = df.loc[~df['remote_est_is_sentinel'], NUMERIC].describe().T
summary['zero_variance'] = summary['std'].fillna(0) == 0
display(summary)

dead = summary.index[summary['zero_variance']].tolist()
if dead:
    print(f'WARNING: no variation in {dead} — a collector was probably stuck.')

## 4 — Data integrity checks

Four things can quietly invalidate the whole evaluation. Check them before
drawing any conclusion from the numbers below.

1. **Fallbacks.** A row with `fell_back=True` was *decided* as EDGE/CLOUD but
   *ran* on the phone. Its `actual_ms` measures local execution plus a wasted
   network timeout, so it belongs in neither the local nor the remote bucket.
2. **Edge→cloud forwarding.** `edge-server` forwards to the cloud whenever its
   `ResourceMonitor` reports overload. That check now reads the container's own
   cgroup budget rather than the host's `/proc`, but forwarding is still legal
   behaviour under genuine load — so `executed_at`, the server's own account of
   where the work ran, remains the authority for attributing latency to a tier.
3. **Errors.** Non-empty `error` means the task failed outright. The CLOUD_ONLY
   baseline does this by design (no fallback), and those rows are recorded
   precisely so the resilience gap is visible rather than missing.
4. **Debug overrides.** Session B forces `remote_energy_mj=50` to make
   `LOW_BATTERY_OFFLOAD` fire. Those rows carry a synthetic estimator value and
   are excluded from cost-model validation in sections 12 and 13.

In [ ]:
n = len(df)
fallbacks = int(df['fell_back'].sum())
errors = int((df['error'].notna() & (df['error'].astype(str) != '')).sum())

print(f'Fallback rate : {fallbacks}/{n} ({100*fallbacks/n:.1f}%)')
print(f'Error rows    : {errors}/{n} ({100*errors/n:.1f}%)')

# Did the tier that ran the task match the tier the policy chose?
remote = df[(~df['fell_back']) & (df['target'].isin(['EDGE', 'CLOUD']))].copy()
remote['agrees'] = remote.apply(
    lambda r: str(r['executed_at']).upper() == str(r['target']).upper(), axis=1
)
mismatch = int((~remote['agrees']).sum())

print(f'\nRemote rows   : {len(remote)}')
print(f'target != executed_at: {mismatch} ({100*mismatch/max(len(remote),1):.1f}%)')

if mismatch:
    print('\nWhere the work actually ran:')
    print(pd.crosstab(remote['target'], remote['executed_at']))
    print(
        '\nNOTE: rows where target=EDGE but executed_at=cloud include an extra\n'
        'edge->cloud hop. Attribute their latency to CLOUD, not EDGE, or the\n'
        'edge numbers will look worse than the edge actually is.'
    )

if 100 * fallbacks / n > 20:
    print('\nWARNING: >20% fallbacks — servers were unreachable for much of the run.')

### Analysis frame

Everything below uses `clean` — successful runs whose measured latency
reflects the tier that actually executed the task. `tier` is the ground-truth
execution location, which is what latency should be attributed to.

In [ ]:
clean = df[
    (~df['fell_back'])
    & (df['error'].isna() | (df['error'].astype(str) == ''))
].copy()

def to_tier(row):
    at = str(row['executed_at']).lower()
    if at in ('edge', 'cloud'):
        return at.upper()
    return 'LOCAL'

clean['tier'] = clean.apply(to_tier, axis=1)
clean['policy'] = clean['rule'].map(
    lambda r: 'Local-only' if r == 'FORCED_LOCAL'
    else 'Cloud-only' if r == 'FORCED_CLOUD'
    else 'ML (RF)' if str(r).startswith('ML_PREDICTED')
    else 'Rule-based'
)

print(f'Clean rows: {len(clean)} / {len(df)}')
print(clean.groupby(['policy', 'tier']).size().unstack(fill_value=0))

# `trusted` additionally drops rows whose estimator outputs were overridden or
# are offline sentinels. Sections 12 and 13 score the cost model, so they may
# only use rows where the estimates are genuine predictions.
trusted = clean[(~clean['has_debug_override']) & (~clean['remote_est_is_sentinel'])]
print(f'\nTrusted rows for cost-model validation: {len(trusted)} / {len(clean)}')

## 5 — Filter to adaptive rows for training

Baseline (`FORCED_*`) and ML (`ML_PREDICTED_*`) rows are excluded from
training: the classifier's job is to reproduce the **rule-based** policy, and
including its own past predictions would be a feedback loop.

In [ ]:
train_df = df[~df['rule'].str.startswith(('FORCED_', 'ML_PREDICTED'))].copy()

print(f'Adaptive rows for training : {len(train_df)}')
print(f'Baseline rows for comparison: {len(df) - len(train_df)}')

per_class = train_df['target'].value_counts()
print('\nClass balance:')
print(per_class)
if per_class.min() < 15:
    print(
        f'\nWARNING: smallest class has {per_class.min()} rows. A stratified '
        '5-fold CV needs >= 5, but anything under ~15 makes the reported '
        'per-class precision/recall very noisy.'
    )

## 6 — Feature engineering

12 features, matching `RandomForestPolicy.FEATURE_ORDER` in the Android app.
**This ordering is a contract** — `RandomForestModelTest` fails the build if the
exported `feature_names` drift from the Kotlin extractor.

Excluded to avoid leakage: `target`, `rule`, `reasoning` (states the decision),
`actual_ms`, `executed_at`, `server_exec_ms` (all measured *after* the decision),
and the identifier columns.

In [ ]:
NETWORK_RANK = {'NONE': 0, 'LTE': 1, 'WIFI': 2, '5G': 3, 'FIVE_G': 3}
COMPLEXITY_RANK = {'LIGHT': 0, 'MEDIUM': 1, 'HEAVY': 2}
TASK_COMPLEXITY = {
    'echo': 0, 'sha256': 0,                        # LIGHT
    'image-grayscale': 1,                          # MEDIUM
    'matrix-multiply': 2, 'video-frame-edges': 2,  # HEAVY
}

# Must stay identical to RandomForestPolicy.FEATURE_ORDER (Kotlin).
FEATURE_ORDER = [
    'battery_percent', 'is_charging', 'network_type_rank', 'network_score',
    'rtt_ms', 'bandwidth_mbps', 'cpu_percent', 'is_stable',
    'task_complexity', 'est_local_ms', 'est_remote_ms', 'speedup',
]

def engineer(d, with_energy=False):
    out = pd.DataFrame(index=d.index)
    out['battery_percent'] = d['battery_percent']
    out['is_charging'] = d['is_charging'].astype(int)
    out['network_type_rank'] = d['network_type'].map(NETWORK_RANK).fillna(0)
    out['network_score'] = d['network_score']
    out['rtt_ms'] = d['rtt_ms']
    out['bandwidth_mbps'] = d['bandwidth_mbps']
    out['cpu_percent'] = d['cpu_percent']
    out['is_stable'] = d['is_stable'].astype(int)
    out['task_complexity'] = d['task_name'].map(TASK_COMPLEXITY).fillna(-1)
    out['est_local_ms'] = d['est_local_ms']
    out['est_remote_ms'] = d['est_remote_ms']
    out['speedup'] = d['speedup']
    out = out[FEATURE_ORDER]
    if with_energy:
        # Section 10 ablation only — NOT part of the deployed feature set.
        out = out.copy()
        out['est_local_energy_mj'] = d['est_local_energy_mj']
        out['est_remote_energy_mj'] = d['est_remote_energy_mj']
        out['energy_ratio'] = (
            d['est_remote_energy_mj'] / d['est_local_energy_mj'].clip(lower=0.001)
        )
    return out

unknown = set(train_df['task_name']) - set(TASK_COMPLEXITY)
if unknown:
    raise ValueError(f'task_name(s) {unknown} have no complexity mapping')

# .map().fillna(0) would silently encode an unmapped network type as NONE,
# i.e. 'offline' — a wrong value the model would happily train on.
unknown_net = set(train_df['network_type']) - set(NETWORK_RANK)
if unknown_net:
    raise ValueError(
        f'network_type(s) {unknown_net} have no rank in NETWORK_RANK. '
        'Add them (and to RandomForestPolicy.NETWORK_RANK) before training.'
    )

X = engineer(train_df)
y = train_df['target']

assert list(X.columns) == FEATURE_ORDER, 'feature order drifted'
print('Features:', list(X.columns))
X.head()

## 7 — Train / test split

Stratified by `target` so every class appears in both halves.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE,
)
print(f'Train: {len(X_train)}   Test: {len(X_test)}')
print(y_train.value_counts(normalize=True).round(3))

## 8 — Random Forest training

- `n_estimators=50`, `max_depth=8` — small enough to export as readable JSON
  and walk in Kotlin in well under a millisecond.
- `class_weight='balanced'` — the rule distribution is uneven by construction.
- `random_state` fixed so the exported model is reproducible.

The **majority-class baseline** is reported alongside accuracy. With a skewed
rule distribution, a classifier that always predicts the most common target can
look strong; accuracy is only meaningful relative to that floor.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=8,
    min_samples_leaf=3,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

train_acc = rf.score(X_train, y_train)
test_acc = rf.score(X_test, y_test)
majority = y_test.value_counts(normalize=True).max()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(rf, X, y, cv=cv, scoring='accuracy', n_jobs=-1)

print(f'Train accuracy      : {train_acc:.3f}')
print(f'Test accuracy       : {test_acc:.3f}')
print(f'5-fold CV accuracy  : {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}')
print(f'Majority-class floor: {majority:.3f}')
print(f'Lift over majority  : {test_acc - majority:+.3f}')

if train_acc - test_acc > 0.15:
    print('\nWARNING: train/test gap > 0.15 — overfitting; reduce max_depth or collect more data.')
if test_acc < 0.85:
    print('\nTest accuracy < 0.85 — per PHASE2_4, do not deploy this model on-device yet.')

## 9 — Classification report and confusion matrix

In [ ]:
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred, digits=3, zero_division=0))

In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title('Counts')
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, cbar=False,
            xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title('Row-normalised (recall per class)')
for ax in axes:
    ax.set_xlabel('Predicted (RF)')
    ax.set_ylabel('Actual (rule-based)')
fig.suptitle('Random Forest vs rule-based policy')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'confusion-matrix.png', dpi=160, bbox_inches='tight')
plt.show()

# Where the two policies disagree is the interesting part for the discussion.
wrong = X_test[y_test.values != y_pred].copy()
wrong['actual_rule_target'] = y_test.values[y_test.values != y_pred]
wrong['rf_predicted'] = y_pred[y_test.values != y_pred]
print(f'\n{len(wrong)} disagreements out of {len(X_test)}:')
display(wrong[['task_complexity', 'network_score', 'battery_percent', 'speedup',
               'actual_rule_target', 'rf_predicted']].head(15))

## 10 — Feature importance, and an energy ablation

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURE_ORDER).sort_values()

fig, ax = plt.subplots(figsize=(7, 5))
importances.plot.barh(ax=ax, color='#1F4E79')
ax.set_xlabel('Importance (mean decrease in Gini impurity)')
ax.set_title('RF feature importance — deployed feature set')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'feature-importance.png', dpi=160, bbox_inches='tight')
plt.show()

print(importances.sort_values(ascending=False).to_string())

### Does the energy estimate help?

The deployed model sees `battery_percent` but never the **energy comparison**
that `LOW_BATTERY_OFFLOAD` actually gates on (`remote_energy < local_energy`).
So it cannot represent that rule exactly — it can only approximate it from
battery level and payload-correlated features.

This ablation quantifies what that costs. It trains a second model on the same
rows with three extra features. **It is not exported** — adding features would
break the `FEATURE_ORDER` contract with the Kotlin extractor. Treat the delta as
evidence for the "future work" section.

In [ ]:
X_energy = engineer(train_df, with_energy=True)
Xe_train, Xe_test, ye_train, ye_test = train_test_split(
    X_energy, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE,
)
rf_energy = RandomForestClassifier(
    n_estimators=50, max_depth=8, min_samples_leaf=3,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1,
).fit(Xe_train, ye_train)

energy_acc = rf_energy.score(Xe_test, ye_test)
energy_cv = cross_val_score(rf_energy, X_energy, y, cv=cv, scoring='accuracy', n_jobs=-1)

ablation = pd.DataFrame({
    'features': [len(FEATURE_ORDER), X_energy.shape[1]],
    'test_accuracy': [test_acc, energy_acc],
    'cv_mean': [cv_scores.mean(), energy_cv.mean()],
    'cv_std': [cv_scores.std(), energy_cv.std()],
}, index=['deployed (12)', '+ energy (15)']).round(3)
display(ablation)

print(f'Energy features change test accuracy by {energy_acc - test_acc:+.3f}')

# How much of the extended model's attention goes to the energy features?
e_imp = pd.Series(rf_energy.feature_importances_, index=X_energy.columns)
share = e_imp[['est_local_energy_mj', 'est_remote_energy_mj', 'energy_ratio']].sum()
print(f'Energy features account for {share:.1%} of total Gini importance.')

## 11 — Latency by policy

Mean alone hides the thing users actually feel. The table reports the median and
the p95 as well: offloading trades a lower median for a heavier tail, and that
trade-off is the substance of the comparison.

Latency is grouped by **`tier`** (where the work truly ran), not by the decided
target — see section 4.

In [ ]:
def latency_stats(g):
    return pd.Series({
        'n': len(g),
        'mean': g['actual_ms'].mean(),
        'median': g['actual_ms'].median(),
        'p95': g['actual_ms'].quantile(0.95),
        'p99': g['actual_ms'].quantile(0.99),
        'std': g['actual_ms'].std(),
    })

by_policy = clean.groupby('policy').apply(latency_stats, include_groups=False).round(1)
display(by_policy)

by_policy_task = (
    clean.groupby(['task_name', 'policy'])
    .apply(latency_stats, include_groups=False)
    .round(1)
)
display(by_policy_task)

In [ ]:
tasks = sorted(clean['task_name'].unique())
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order = [p for p in ['Local-only', 'Cloud-only', 'Rule-based', 'ML (RF)']
         if p in clean['policy'].unique()]

sns.barplot(data=clean, x='task_name', y='actual_ms', hue='policy',
            hue_order=order, errorbar=('ci', 95), ax=axes[0])
axes[0].set_title('Mean latency with 95% CI')
axes[0].set_ylabel('actual_ms')
axes[0].tick_params(axis='x', rotation=30)

p95 = (clean.groupby(['task_name', 'policy'])['actual_ms']
       .quantile(0.95).reset_index())
sns.barplot(data=p95, x='task_name', y='actual_ms', hue='policy',
            hue_order=order, ax=axes[1])
axes[1].set_title('p95 latency (tail)')
axes[1].set_ylabel('actual_ms (p95)')
axes[1].tick_params(axis='x', rotation=30)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'latency-by-policy.png', dpi=160, bbox_inches='tight')
plt.show()

### Is the difference real, or sampling noise?

Latency distributions are right-skewed, so a t-test's normality assumption does
not hold. A **bootstrap confidence interval** on the difference in means makes no
distributional assumption: resample each group with replacement 10,000 times and
read off the 2.5th and 97.5th percentiles. If the interval excludes zero, the
difference is significant at the 5% level.

In [ ]:
def bootstrap_diff(a, b, n_boot=10_000, seed=RANDOM_STATE):
    # 95% CI for mean(a) - mean(b), by percentile bootstrap.
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 2 or len(b) < 2:
        return np.nan, np.nan, np.nan
    r = np.random.default_rng(seed)
    diffs = (r.choice(a, (n_boot, len(a)), replace=True).mean(axis=1)
             - r.choice(b, (n_boot, len(b)), replace=True).mean(axis=1))
    return a.mean() - b.mean(), *np.percentile(diffs, [2.5, 97.5])

rows = []
policies = clean['policy'].unique()
if 'Rule-based' in policies:
    for other in [p for p in policies if p != 'Rule-based']:
        for task in tasks:
            sub = clean[clean['task_name'] == task]
            d, lo, hi = bootstrap_diff(
                sub.loc[sub['policy'] == 'Rule-based', 'actual_ms'],
                sub.loc[sub['policy'] == other, 'actual_ms'],
            )
            rows.append({
                'task': task, 'vs': other, 'mean_diff_ms': d,
                'ci_low': lo, 'ci_high': hi,
                'significant': bool(np.isfinite(lo) and (lo > 0 or hi < 0)),
            })

sig = pd.DataFrame(rows).round(1)
print('Rule-based minus <other>; negative = rule-based is faster.')
display(sig)

## 12 — Are the estimators any good?

`LatencyEstimator` drives every decision, but its accuracy has never been
measured — the policy could be making correct decisions from bad numbers, or
bad decisions from good ones, and the rule distribution alone cannot tell them
apart.

For each row we compare the estimate the policy *used* against the latency that
was *measured*, and report MAE, MAPE, and bias (mean signed error — positive
means the estimator is systematically pessimistic).

In [ ]:
# `trusted` excludes offline sentinels and debug-overridden rows — neither
# carries a genuine estimate, so scoring the estimator against them is meaningless.
est = trusted.copy()
est['estimated_ms'] = np.where(
    est['tier'] == 'LOCAL', est['est_local_ms'], est['est_remote_ms']
)
est['error_ms'] = est['estimated_ms'] - est['actual_ms']
est['abs_pct_error'] = (est['error_ms'].abs() / est['actual_ms'].clip(lower=1)) * 100

def acc_stats(g):
    return pd.Series({
        'n': len(g),
        'MAE_ms': g['error_ms'].abs().mean(),
        'MAPE_%': g['abs_pct_error'].mean(),
        'bias_ms': g['error_ms'].mean(),
        'corr': g['estimated_ms'].corr(g['actual_ms']),
    })

print('Estimator accuracy by execution tier:')
display(est.groupby('tier').apply(acc_stats, include_groups=False).round(2))
print('\nBy task:')
display(est.groupby(['task_name', 'tier']).apply(acc_stats, include_groups=False).round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(data=est, x='estimated_ms', y='actual_ms', hue='tier',
                style='task_name', alpha=0.7, ax=axes[0])
lim = max(est['estimated_ms'].max(), est['actual_ms'].max()) * 1.05
axes[0].plot([0, lim], [0, lim], 'k--', lw=1, label='perfect estimate')
axes[0].set(xlim=(0, lim), ylim=(0, lim),
            xlabel='estimated ms (used for the decision)',
            ylabel='actual ms (measured)',
            title='Predicted vs actual latency')
axes[0].legend(fontsize=7, loc='upper left')

sns.boxplot(data=est, x='tier', y='error_ms', ax=axes[1])
axes[1].axhline(0, color='k', ls='--', lw=1)
axes[1].set(ylabel='estimate - actual (ms)',
            title='Estimator bias (above 0 = pessimistic)')

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'estimator-accuracy.png', dpi=160, bbox_inches='tight')
plt.show()

### Splitting network cost from compute cost

`server_exec_ms` is the server's own measurement of its handler. Subtracting it
from the phone's wall-clock isolates the network overhead — the quantity the
whole offloading decision hinges on, and the one the cost model approximates
with `rtt + payload/bandwidth`.

In [ ]:
remote_rows = clean[clean['tier'].isin(['EDGE', 'CLOUD'])].copy()
if len(remote_rows) and remote_rows['server_exec_ms'].notna().any():
    remote_rows['network_overhead_ms'] = (
        remote_rows['actual_ms'] - remote_rows['server_exec_ms']
    )
    overhead = remote_rows.groupby(['tier', 'task_name']).agg(
        n=('network_overhead_ms', 'size'),
        server_ms=('server_exec_ms', 'mean'),
        network_ms=('network_overhead_ms', 'mean'),
        total_ms=('actual_ms', 'mean'),
        measured_rtt=('rtt_ms', 'mean'),
    ).round(1)
    overhead['network_share_%'] = (
        100 * overhead['network_ms'] / overhead['total_ms']
    ).round(1)
    display(overhead)
    print(
        '\nIf network_share is large for HEAVY tasks, the compute floor of 1.5x\n'
        'in OffloadingPolicy is too permissive for this deployment.'
    )
else:
    print('No remote rows with server_exec_ms — collect data with servers reachable.')

## 13 — Validating the energy model against measurement

`Energy_Local = P_cpu x T_local` uses a fixed 800 mW CPU coefficient that
`EnergyEstimator` itself calls an "order-of-magnitude estimate". Approximate
power models of that kind have been reported in the literature with errors
ranging from 40% to over 200%, so an unvalidated coefficient is a real liability.

`measured_power_mw` and `measured_energy_mj` come from Android's
`BATTERY_PROPERTY_CURRENT_NOW`, sampled either side of each run (`P = V x I`).
Three caveats belong in the thesis beside any number below:

1. It is **whole-device** draw — screen, radio, background work included — so it
   is an upper bound on the task's cost, not an attribution to the task.
2. The platform updates battery current at roughly 1 Hz, so short tasks are
   measured poorly. Rows under 500 ms are filtered out.
3. Not every device implements the property. Where it is absent the column is
   empty and the model simply cannot be validated on that hardware — say so
   rather than reporting modelled energy as though it were measured.

What this *can* establish: whether modelled energy **tracks** measured energy,
and what CPU coefficient the data actually implies.

In [ ]:
has_power = trusted['measured_energy_mj'].notna().any()

if not has_power:
    print(
        'No measured_energy_mj in this dataset.\n'
        'Either the device does not implement BATTERY_PROPERTY_CURRENT_NOW, or the\n'
        'data predates power sampling. Report the energy model as unvalidated.'
    )
else:
    MIN_MS_FOR_POWER = 500   # below this the ~1 Hz sampling dominates
    pw = trusted[trusted['measured_energy_mj'].notna()
                 & (trusted['actual_ms'] >= MIN_MS_FOR_POWER)]
    print(f'Rows with measured energy and >= {MIN_MS_FOR_POWER}ms duration: {len(pw)}')

    local_pw = pw[pw['tier'] == 'LOCAL']
    if len(local_pw) >= 10:
        corr = local_pw['est_local_energy_mj'].corr(local_pw['measured_energy_mj'])
        ratio = local_pw['measured_energy_mj'] / local_pw['est_local_energy_mj'].clip(lower=1e-6)
        print(f'\nLocal execution, n={len(local_pw)}')
        print(f'  correlation(modelled, measured) : {corr:.3f}')
        print(f'  measured/modelled ratio  median : {ratio.median():.2f}')
        print(f'                            IQR   : {ratio.quantile(.25):.2f} - {ratio.quantile(.75):.2f}')

        implied_mw = (local_pw['measured_energy_mj'] * 1000 / local_pw['actual_ms']).median()
        print('\n  Model assumes CPU_POWER_MW = 800')
        print(f'  Median measured whole-device draw: {implied_mw:.0f} mW')
        verdict = 'plausible' if 400 <= implied_mw <= 2500 else 'questionable'
        print(f'  => 800 mW is {verdict} as the CPU share of that total.')
    else:
        print('\nToo few local rows with usable power samples to estimate a coefficient.')

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    sns.scatterplot(data=pw, x='est_local_energy_mj', y='measured_energy_mj',
                    hue='task_name', style='tier', alpha=0.75, ax=axes[0])
    lim = pw[['est_local_energy_mj', 'measured_energy_mj']].max().max() * 1.05
    axes[0].plot([0, lim], [0, lim], 'k--', lw=1, label='1:1')
    axes[0].set(xlabel='modelled local energy (mJ)', ylabel='measured energy (mJ)',
                title='Energy model vs battery-sensor measurement')
    axes[0].legend(fontsize=7)

    sns.scatterplot(data=pw, x='actual_ms', y='measured_power_mw',
                    hue='tier', alpha=0.75, ax=axes[1])
    axes[1].axhline(800, color='r', ls='--', lw=1, label='model CPU_POWER_MW = 800')
    axes[1].set(xlabel='task duration (ms)', ylabel='measured device power (mW)',
                title='Measured draw vs the modelled coefficient')
    axes[1].legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'energy-validation.png', dpi=160, bbox_inches='tight')
    plt.show()

### Does the model at least rank the options correctly?

A miscalibrated model is still useful if it gets the *comparison* right — the
policy only asks "is remote cheaper than local?", never "how many mJ exactly?".
A constant-factor error cancels in that comparison; a task-dependent one does not.

In [ ]:
energy = trusted.copy()
energy['remote_cheaper'] = energy['est_remote_energy_mj'] < energy['est_local_energy_mj']
energy['offloaded'] = energy['tier'] != 'LOCAL'
print('Energy-optimal choice vs actual choice (trusted rows only):')
print(pd.crosstab(energy['remote_cheaper'], energy['offloaded'],
                  rownames=['remote cheaper'], colnames=['offloaded']))

low_batt = energy[(energy['battery_percent'] < 30) & (~energy['is_charging'])]
if len(low_batt):
    took = low_batt.loc[low_batt['remote_cheaper'], 'offloaded'].mean()
    print(f'\nBelow 30% battery where offloading saves energy: '
          f'offloaded {took:.0%} of the time (n={int(low_batt["remote_cheaper"].sum())}).')

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.scatterplot(data=energy, x='est_local_energy_mj', y='est_remote_energy_mj',
                hue='target', style='task_name', alpha=0.7, ax=ax)
lim = energy[['est_local_energy_mj', 'est_remote_energy_mj']].max().max() * 1.05
ax.plot([0, lim], [0, lim], 'k--', lw=1)
ax.set(xlabel='modelled local energy (mJ)', ylabel='modelled remote energy (mJ)',
       title='Below the line = model says offloading saves energy')
ax.legend(fontsize=7)
fig.tight_layout()
plt.show()

## 13b — Payload sensitivity

Session J sweeps payload size *within* a task type. That is what separates the
transmission term (`payload / bandwidth`) from the compute term — with a single
fixed size per task the two are perfectly confounded and neither can be
attributed.

In [ ]:
sizes = trusted.groupby('task_name')['input_size_bytes'].nunique()
varied = sizes[sizes > 1].index.tolist()

if not varied:
    print(
        'Every task ran at one payload size, so the transmission term cannot be\n'
        'separated from compute. Run Session J.'
    )
else:
    print(f'Tasks with varied payload size: {varied}\n')
    sub = trusted[trusted['task_name'].isin(varied)]
    for tier in ['LOCAL', 'EDGE', 'CLOUD']:
        t = sub[sub['tier'] == tier]
        if len(t) < 10:
            continue
        r = t['input_size_bytes'].corr(t['actual_ms'])
        print(f'{tier:6} n={len(t):4}  corr(payload_bytes, actual_ms) = {r:+.3f}')
    print(
        '\nExpect a clearly positive correlation for EDGE/CLOUD (the bytes cross a\n'
        'network) and a weaker one for LOCAL. If LOCAL correlates as strongly, the\n'
        'tasks are compute-bound in payload size and the sweep is not isolating\n'
        'transmission cost.'
    )

    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.scatterplot(data=sub, x='input_size_bytes', y='actual_ms',
                    hue='tier', style='task_name', alpha=0.75, ax=ax)
    ax.set(xscale='log', yscale='log',
           xlabel='payload (bytes, log)', ylabel='actual_ms (log)',
           title='Latency vs payload size by execution tier')
    ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'payload-sensitivity.png', dpi=160, bbox_inches='tight')
    plt.show()

## 13c — Mobility

Session I forces STATIONARY / WALKING / VEHICLE. `pickRemoteTarget` routes to
EDGE only when the device is stationary, and `LatencyEstimator` adds up to
200 ms of mobility penalty on the same signal. Without a forced sweep a phone on
a desk reports STATIONARY on every row and neither path is ever observed.

In [ ]:
if trusted['is_stable'].nunique() < 2:
    print(
        'Every row reports is_stable=True, so the mobility branch of pickRemoteTarget\n'
        'was never exercised. Run Session I, or drop mobility from the claims.'
    )
else:
    print('Target chosen by mobility state:')
    print(pd.crosstab(trusted['is_stable'], trusted['target'],
                      rownames=['is_stable'], colnames=['target']))

    print('\nLatency by mobility state and tier:')
    print(trusted.groupby(['is_stable', 'tier'])['actual_ms']
          .agg(['size', 'mean', 'median']).round(1).to_string())

    stable = trusted[trusted['is_stable']]
    moving = trusted[~trusted['is_stable']]
    if len(stable) and len(moving):
        print(f'\nEDGE share when stationary : {(stable["tier"] == "EDGE").mean():.1%}')
        print(f'EDGE share when moving     : {(moving["tier"] == "EDGE").mean():.1%}')
        print('The policy should prefer EDGE only when stationary. A similar share in '
              'both means the rule is not actually discriminating.')

## 14 — When does the system fall back?

The fallback rate is the resilience number. Broken down by condition it becomes
diagnostic: fallbacks concentrated at low `network_score` mean the
`UNSTABLE_NETWORK` threshold of 0.30 is set too low to protect the user.

In [ ]:
fb = df[df['target'].isin(['EDGE', 'CLOUD'])].copy()
if len(fb):
    fb['net_bucket'] = pd.cut(fb['network_score'], [-0.01, 0.3, 0.6, 1.01],
                              labels=['<0.30 (unstable)', '0.30-0.60', '>=0.60 (good)'])
    fb['rtt_bucket'] = pd.cut(fb['rtt_ms'], [-1, 50, 150, 400, 1e9],
                              labels=['<50ms', '50-150ms', '150-400ms', '>400ms'])

    for col, title in [('net_bucket', 'network score'), ('rtt_bucket', 'RTT'),
                       ('network_type', 'network type'), ('target', 'chosen target')]:
        t = fb.groupby(col, observed=True)['fell_back'].agg(['size', 'sum', 'mean'])
        t.columns = ['attempts', 'fallbacks', 'rate']
        t['rate'] = (100 * t['rate']).round(1)
        print(f'\nFallback rate by {title}:')
        print(t.to_string())
else:
    print('No remote attempts in this dataset.')

## 15 — Approximate regret against a matched-condition oracle

**This is the answer to the sharpest question a reviewer can ask.** Sections 8–10
measure how well the RF *imitates the rules*. Neither says whether the rules were
right. Accuracy against rule-generated labels is circular: a model scoring 1.00
has perfectly reproduced a policy that might itself be wrong.

There is no true oracle here — each task ran under exactly one target, so the
counterfactual ("what would EDGE have cost for *this* run?") was never observed.
What we can build is a **matched-condition oracle**: bucket rows by task and by
discretised context, and within each bucket treat the empirically fastest tier as
the choice an oracle would have made. Regret is then

    regret(row) = actual_ms(row) - mean actual_ms of the best tier in its bucket

Averaged per policy, this estimates how much latency each policy leaves on the
table. Caveats, which belong in the thesis: buckets with a missing tier are
dropped, bucketing is coarse, and within-bucket variance is treated as noise.

In [ ]:
oracle_src = clean.copy()
oracle_src['net_b'] = pd.cut(oracle_src['network_score'], [-0.01, 0.3, 0.6, 1.01],
                             labels=['low', 'mid', 'high'])
oracle_src['batt_b'] = pd.cut(oracle_src['battery_percent'], [-1, 30, 70, 101],
                              labels=['low', 'mid', 'high'])
oracle_src['cpu_b'] = pd.cut(oracle_src['cpu_percent'], [-1, 40, 101],
                             labels=['idle', 'busy'])

KEYS = ['task_name', 'net_b', 'batt_b', 'cpu_b']

bucket_tier = (
    oracle_src.groupby(KEYS + ['tier'], observed=True)['actual_ms']
    .agg(['mean', 'size']).reset_index()
)
# Require a few samples before trusting a bucket/tier mean.
bucket_tier = bucket_tier[bucket_tier['size'] >= 3]

# Only buckets where at least two tiers were observed support a comparison.
counts = bucket_tier.groupby(KEYS, observed=True)['tier'].nunique()
usable = counts[counts >= 2].index

best = (
    bucket_tier.set_index(KEYS)
    .loc[bucket_tier.set_index(KEYS).index.isin(usable)]
    .reset_index()
    .sort_values('mean')
    .groupby(KEYS, observed=True)
    .first()
    .rename(columns={'mean': 'best_ms', 'tier': 'best_tier'})[['best_ms', 'best_tier']]
)

print(f'Buckets with >=2 observed tiers: {len(best)} of {counts.size}')

scored = oracle_src.merge(best, left_on=KEYS, right_index=True, how='inner')
scored['regret_ms'] = scored['actual_ms'] - scored['best_ms']
scored['chose_best'] = scored['tier'] == scored['best_tier']

if len(scored):
    regret = scored.groupby('policy').agg(
        n=('regret_ms', 'size'),
        mean_regret_ms=('regret_ms', 'mean'),
        median_regret_ms=('regret_ms', 'median'),
        p95_regret_ms=('regret_ms', lambda s: s.quantile(0.95)),
        picked_best_tier=('chose_best', 'mean'),
    ).round(2)
    regret['picked_best_tier'] = (100 * regret['picked_best_tier']).round(1)
    display(regret)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.barplot(data=scored, x='policy', y='regret_ms', errorbar=('ci', 95), ax=ax,
                order=[p for p in order if p in scored['policy'].unique()])
    ax.axhline(0, color='k', ls='--', lw=1)
    ax.set(ylabel='regret vs matched-condition oracle (ms)',
           title='Lower is better; 0 = matched the best observed tier')
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'regret-by-policy.png', dpi=160, bbox_inches='tight')
    plt.show()

    print('\nOracle tier choice by bucket:')
    print(best['best_tier'].value_counts().to_string())
else:
    print(
        'Not enough overlapping coverage to estimate regret.\n'
        'Each (task, network, battery, cpu) bucket needs >=3 rows for >=2 tiers.\n'
        'Re-run sessions F and G under the same conditions as the adaptive sessions.'
    )

## 16 — Export the model for on-device deployment

Serialises each tree to portable JSON so the Kotlin runtime can predict without
an ML framework. `feature_names` is written from `FEATURE_ORDER`; the Android
side asserts it matches on load and `RandomForestModelTest` asserts it at build
time.

In [ ]:
def tree_to_dict(tree):
    t = tree.tree_
    values = t.value.reshape(-1, len(rf.classes_))
    values = values / values.sum(axis=1, keepdims=True).clip(min=1e-12)
    return {
        'feature':   t.feature.astype(int).tolist(),
        'threshold': t.threshold.astype(float).tolist(),
        'left':      t.children_left.astype(int).tolist(),
        'right':     t.children_right.astype(int).tolist(),
        'value':     values.astype(float).tolist(),
    }

model_json = {
    'feature_names': FEATURE_ORDER,
    'classes': list(rf.classes_),
    'network_rank': NETWORK_RANK,
    'task_complexity': TASK_COMPLEXITY,
    # Keyed by complexity LEVEL, not task name. RandomForestPolicy compares
    # its own encoding against this; the task_complexity map above uses a
    # different key space and cannot be compared directly.
    'complexity_rank': COMPLEXITY_RANK,
    'trees': [tree_to_dict(t) for t in rf.estimators_],
}

model_path = OUTPUT_DIR / 'rf-model.json'
model_path.write_text(json.dumps(model_json), encoding='utf-8')

size_kb = model_path.stat().st_size / 1024
print(f'Exported {model_path.name} - {size_kb:.1f} KB, '
      f'{len(model_json["trees"])} trees, classes {model_json["classes"]}')
if size_kb > 100:
    print('WARNING: >100 KB exceeds the size budget in PHASE2_4.')

### Round-trip check

The exported JSON must reproduce sklearn's predictions **exactly** on every test
row. A partial check would hide the boundary cases (`<=` vs `<`) that are the
most likely source of a port bug.

In [ ]:
def predict_with_json(model, x_row):
    votes = np.zeros(len(model['classes']))
    for tree in model['trees']:
        node = 0
        while tree['feature'][node] >= 0:
            f = tree['feature'][node]
            node = (tree['left'][node] if x_row[f] <= tree['threshold'][node]
                    else tree['right'][node])
        votes += np.array(tree['value'][node])
    return model['classes'][int(np.argmax(votes))]

json_preds = [predict_with_json(model_json, row) for row in X_test.to_numpy()]
sk_preds = list(rf.predict(X_test))
matches = sum(j == s for j, s in zip(json_preds, sk_preds))

print(f'JSON model matches sklearn on {matches}/{len(sk_preds)} test rows')
if matches != len(sk_preds):
    raise AssertionError('Exported model diverges from sklearn — do not deploy.')
print('Safe to copy to mobile/app/src/main/assets/rf-model.json')

## 17 — Summary for the thesis chapter

In [ ]:
lines = [
    '# RF vs rule-based — evaluation summary',
    '',
    f'Generated from `{DATA_PATH.name}` — {len(df)} rows '
    f'({len(clean)} clean, {len(train_df)} adaptive).',
    '',
    '## Classifier (section 5.3)',
    '',
    f'- Test accuracy: **{test_acc:.3f}** (majority-class floor {majority:.3f}, '
    f'lift {test_acc - majority:+.3f})',
    f'- 5-fold CV: **{cv_scores.mean():.3f} ± {cv_scores.std():.3f}**',
    f'- Train accuracy: {train_acc:.3f} (gap {train_acc - test_acc:+.3f})',
    '',
    '### Top-5 features',
    *[f'- {n}: {v:.3f}' for n, v in importances.sort_values(ascending=False).head(5).items()],
    '',
    f'Energy ablation: {energy_acc:.3f} with energy features vs {test_acc:.3f} without '
    f'({energy_acc - test_acc:+.3f}).',
    '',
    '## Data integrity (section 5.1)',
    '',
    f'- Fallback rate: {100*fallbacks/n:.1f}%',
    f'- target != executed_at: {mismatch}/{len(remote)} remote rows',
    f'- Rows with debug overrides (excluded from cost-model validation): {n_override}',
    '',
    '## Latency by policy (section 5.5)',
    '',
    '```',
    by_policy.to_string(),
    '```',
    '',
    '## Estimator accuracy (section 5.4)',
    '',
    '```',
    est.groupby('tier').apply(acc_stats, include_groups=False).round(2).to_string(),
    '```',
]

if len(scored):
    lines += [
        '',
        '## Regret vs matched-condition oracle (section 5.6)',
        '',
        '```',
        regret.to_string(),
        '```',
        '',
        'Accuracy against rule-generated labels is circular — it measures imitation, '
        'not decision quality. Regret is the non-circular number.',
    ]

out = OUTPUT_DIR / 'comparison-summary.md'
out.write_text('\n'.join(lines), encoding='utf-8')
print(f'Wrote {out}')
print('\n'.join(lines))